<a href="https://colab.research.google.com/github/andreagrioni/Tutorials/blob/master/Generate_train_data_mirna_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pipeline Functions

In [0]:
import time
import numpy as np
import pandas as pd

## Generate 2d Matrix

The function generates an array that correspond to the 2d matrix used as input  

for Custard Model. The function can also create a 1d tensor for both bind and  

mirna sequences by setting the `aux` funtion paramenter to `True`.

In [0]:
def one_hot_encoding(df, tensor_dim=(50,20,1), aux=False, log=False):
    """
    fun transform input database to
    one hot encoding array.
    paramenters:
    df=input table as pandas dataframe
    tensor_dim= 2d matrix shape
    aux=create inputs for LSTM
    log=print log
    """

    # reset df indexes (needed for multithreading)
    df.reset_index(inplace=True, drop=True)
    
    # alphabet for watson-crick interactions.
    alphabet = {"AT": 1., "TA": 1., "GC": 1., "CG": 1.} 
    # one hot encoding of nt sequences for conv1d + LSTM input
    nt_pos_voc = {
        "A" : np.array([1.,0.,0.,0.]),
        "T" : np.array([0.,1.,0.,0.]),
        "C" : np.array([0.,0.,1.,0.]),
        "G" : np.array([0.,0.,0.,1.]),
        "N" : np.array([0.25,0.25,0.25,0.25]),
    
    }

    # labels to one hot encoding
    labels = np.where(df.label == 'positive', 1., 0.)
    
    if aux == True: # conv1d + LSTM input
        bind_matrix_l = list()
        mirna_matrix_l = list()
    else: 
        bind_matrix, mirna_matrix = None, None

    # create empty main 2d matrix array
    N = df.shape[0] # number of samples in df
    shape_matrix_2d = (N, *tensor_dim) # 2d matrix shape 
    ohe_matrix_2d = np.zeros(shape_matrix_2d, dtype="float32")

    start = time.time()

    for index, row in df.iterrows():        
        if aux:
            bind_matrix_l.append(
                [nt_pos_voc[letter] for letter in row.binding_sequence.upper()]
                                  )
            mirna_matrix_l.append(
                [nt_pos_voc[letter] for letter in row.mirna_binding_sequence.upper()]
                    )

        for bind_index, bind_nt in enumerate(row.binding_sequence.upper()):
                
            for mirna_index, mirna_nt in enumerate(
                row.mirna_binding_sequence.upper()
            ):

                pair = bind_nt + mirna_nt
                ohe_matrix_2d[index, bind_index, mirna_index, 0] = alphabet.get(pair, 0)
                
        if index % 1000 == 0 and log==True: # write something
            end = time.time()
            print(
                "rows:\t%s" % (index),
                "elapsed (sec):\t%s" % (end - start),
                sep=" | ",
            )

    if aux:
        bind_matrix = np.array(bind_matrix_l)
        mirna_matrix = np.array(mirna_matrix_l)
    if aux:
      return (ohe_matrix_2d, bind_matrix, mirna_matrix, labels)
    else:
      return (ohe_matrix_2d, labels)

## General file management functions

In [0]:
import joblib

def save_joblib(object_, filepath):
    joblib.dump(object_, filepath)
    return filepath

def load_joblib(filepath):
    return joblib.load(filepath)

## Parallelized conversion of an array/dataframe to 2D matrix

The below function takes as input a Pandas df or numpy array, and split it into  batches for parallelization.

Usage:

`output = multithread(df, one_hot_encoding, aux=False, log=False, n_cores=24)`  
`data = join_cores_results(output, aux=True)`

In [0]:
def join_cores_results(multithread_output, aux=False):
  """ join the output of different core processes """
  if aux:
    array_2d_matrix = np.concatenate(
        [ process[0] for process in multithread_output ]
        )
    array_bind_seq = np.concatenate(
        [ process[1] for process in multithread_output ]
        )
    array_micro_seq = np.concatenate(
        [ process[2] for process in multithread_output ]
        )
    array_labels = np.concatenate(
        [ process[3] for process in multithread_output ]
        )
    return (array_2d_matrix, array_bind_seq,
            array_micro_seq, array_labels)
  else:
    array_2d_matrix = np.concatenate(
        [ process[0] for process in multithread_output ]
        )
    array_labels = np.concatenate(
        [ process[1] for process in multithread_output ]
        )
    return (array_2d_matrix, array_labels)

In [0]:
from multiprocessing import Pool
from functools import partial

def multithread(df, func, aux=False, log=False, n_cores=4):
    iterable = np.array_split(df, n_cores)
    pool = Pool(n_cores)
    lock_func = partial(func, aux=aux, log=log)
    df_update = pool.map(lock_func, iterable)
    pool.close()
    pool.join()
    data = join_cores_results(df_update, aux=aux)
    return data

## Create Negative Class Shuffle

The function generates the negative class by creating a connection between each

binding site and all mirna (expect the real one). If argument `mirna_dict` is

provided as dictionary of mirna sequences, this dictionary will be used to

create the negative class. Otherwise, all unique mirna sequences of the input

df will be used to generate samples for the negative class.


In [0]:
def negative_class_shuffle(df, mirna_dict=None, neg_ratio=None):
    if not mirna_dict:
        # generate mirna db of unique sequences
        mirnadb = pd.DataFrame(
            df.mirna_binding_sequence.unique(), columns=['mirnaid']
        )
    else:
        mirnadb = pd.DataFrame(mirna_dict)
        mirnadb.columns = ['mirnaid']
    # add mirna db to each row of df
    connections = mirnadb.assign(key=1).merge(
          df.assign(key=1), on='key'
          ).drop(['key', 'label'],axis=1)
    # find index of positive connection
    positive_samples_mask = (connections.mirnaid == 
                             connections.mirna_binding_sequence)
    # drop positive connection to create negative samples
    negative_df = connections[~positive_samples_mask].copy().drop(
      ['mirna_binding_sequence'], axis=1
      ).reset_index(drop=True)
    # rename cols
    negative_df.columns = ['mirna_binding_sequence', 'binding_sequence']
    # add negative labels
    negative_df['label'] = 'negative'
    if neg_ratio == None:
        return negative_df
    else:
        neg_samples = int(df.shape[0] * neg_ratio)
        return negative_df.sample(n = neg_samples)

# Running Pipeline

## generate dummy dataset

In [0]:
# test df
import pandas as pd
import numpy as np

dummy_set = {
    "binding_sequence" : ["ATGGG" * 10] + ["AAAAA" * 10],
    "mirna_binding_sequence" : ["ACCC" * 5] + ["TTTT" * 5],
    "label" : ["positive"] * 2
    }
positive_samples = pd.DataFrame(dummy_set)

## create negative class

In [0]:
negative_samples = negative_class_shuffle(positive_samples)

## df to one hot encoding

Convert sequences to one hot encoding.

### no auxiliary inputs

generate only 2d matrix and labels.

In [0]:
positive_data = one_hot_encoding(
    positive_samples, tensor_dim=(50,20,1), aux=False, log=False)
negative_data = one_hot_encoding(
    negative_samples, tensor_dim=(50,20,1), aux=False, log=False)

### with auxiliary inputs

generate 2d matrix, 1d bind sequence, 1d micro sequence, labels

In [0]:
positive_data_with_aux = one_hot_encoding(
    positive_samples, tensor_dim=(50,20,1), aux=True, log=False)
negative_data_with_aux = one_hot_encoding(
    negative_samples, tensor_dim=(50,20,1), aux=True, log=False)

### df to one hot encoding with multithread

Here an example of how to run generate the same datasets 

in multithread settings.

In [0]:
data = multithread(
    positive_samples, one_hot_encoding, aux=True, log=False, n_cores=2
    )